product week panel

In [3]:
import pandas as pd, numpy as np

clean = pd.read_parquet("../data/processed_clean.parquet")

# 1. Remove line-level price errors (>5x or <0.2x the product's median price)
med = clean.groupby("StockCode")["Price"].transform("median")
ratio = clean["Price"] / med
print("Price-error rows:", ((ratio > 5) | (ratio < 0.2)).sum())
clean = clean[(ratio <= 5) & (ratio >= 0.2)].copy()

# 2. Product-week panel with a mix-robust price
panel = (clean.groupby(["StockCode", "Week"])
         .agg(units=("Quantity", "sum"),
              revenue=("Revenue", "sum"),
              n_lines=("Invoice", "size"),
              price=("Price", "median"),
              min_price=("Price", "min"),
              max_price=("Price", "max"))
         .reset_index())
panel["spread"] = panel["max_price"] / panel["min_price"]

# 3. Keep weeks with a single clear price point
print("Share of weeks with spread <= 1.10:", (panel["spread"] <= 1.10).mean().round(3))
panel = panel[panel["spread"] <= 1.10].copy()
print(panel.shape)

Price-error rows: 3045
Share of weeks with spread <= 1.10: 0.581
(114303, 9)


keep only products with enough history and enough price movement

In [4]:
stats = panel.groupby("StockCode").agg(
    weeks=("Week", "nunique"),
    price_levels=("price", lambda s: s.round(2).nunique()),
    price_cv=("price", lambda s: s.std() / s.mean()))

eligible = stats[(stats["weeks"] >= 20) & (stats["price_levels"] >= 3) & (stats["price_cv"] >= 0.03)]
print("Eligible products:", len(eligible), "of", len(stats))
panel = panel[panel["StockCode"].isin(eligible.index)].copy()
print(panel.shape)

Eligible products: 1700 of 4826
(66694, 9)


In [5]:
print(panel.duplicated(["StockCode", "Week"]).sum())      # should be 0
print(panel["units"].describe())
print(panel.groupby("StockCode")["Week"].nunique().describe())

0
count    66694.000000
mean        14.585990
std         32.373601
min          1.000000
25%          2.000000
50%          5.000000
75%         13.000000
max       1153.000000
Name: units, dtype: float64
count    1700.000000
mean       39.231765
std        12.562702
min        20.000000
25%        29.000000
50%        38.000000
75%        48.000000
max        76.000000
Name: Week, dtype: float64


categories, features, and margin assumption

In [10]:
import re

rules = {   # order = priority
    "Christmas & Seasonal": ["CHRISTMAS", "XMAS", "SNOWMAN", "ADVENT", "SANTA", "REINDEER", "EASTER", "HALLOWEEN"],
    "Paper & Party":        ["PAPER", "CARD", "WRAP", "PARTY", "BUNTING", "BALLOON", "NAPKIN", "RIBBON", "GIFT"],
    "Lighting & Candles":   ["LIGHT", "LAMP", "CANDLE", "T-LIGHT", "LANTERN", "HOLDER"],
    "Kitchen & Dining":     ["MUG", "CUP", "PLATE", "BOWL", "JAR", "TEAPOT", "CAKE", "KITCHEN", "SPOON", "TEA", "COASTER"],
    "Bags & Storage":       ["BAG", "BOX", "BASKET", "TIN", "STORAGE", "CASE", "TRAY"],
    "Jewellery & Accessories": ["NECKLACE", "BRACELET", "EARRINGS", "RING", "CHARM", "BEADS", "SCARF", "PURSE"],
    "Toys & Kids":          ["TOY", "DOLL", "GAME", "CHILDRENS", "KIDS", "PUZZLE", "SPACEBOY", "DOLLY"],
    "Stationery":           ["NOTEBOOK", "PENCIL", "PEN", "STICKER", "POSTCARD", "ERASER", "STAMP"],
    "Garden & Outdoor":     ["GARDEN", "PLANT", "FLOWER", "BIRD", "WATERING", "POT"],
    "Home Decor":           ["FRAME", "CUSHION", "SIGN", "CLOCK", "HEART", "DECORATION", "MIRROR", "DOOR", "HOOK", "HANGING"],
}
compiled = {c: re.compile(r"\b(?:" + "|".join(map(re.escape, w)) + r")S?\b") for c, w in rules.items()}

def categorize(d):
    for cat, pat in compiled.items():
        if pat.search(d):
            return cat
    return "Other"

panel = panel.drop(columns=["category"])
cat_map = desc.apply(categorize).rename("category")
panel = panel.merge(cat_map, left_on="StockCode", right_index=True)

print(panel.shape)   # still 66,694 rows
print(panel.groupby("category")["StockCode"].nunique().sort_values(ascending=False))

(66694, 19)
category
Other                      516
Home Decor                 216
Kitchen & Dining           178
Lighting & Candles         168
Bags & Storage             160
Paper & Party              138
Garden & Outdoor            99
Stationery                  67
Jewellery & Accessories     67
Christmas & Seasonal        63
Toys & Kids                 28
Name: StockCode, dtype: int64


Model features:

In [11]:
panel["log_units"] = np.log(panel["units"])
panel["log_price"] = np.log(panel["price"])
panel["month"] = panel["Week"].dt.month
panel["week_of_year"] = panel["Week"].dt.isocalendar().week.astype(int)
panel["is_q4"] = panel["month"].isin([10, 11, 12]).astype(int)
panel["price_index"] = panel["price"] / panel.groupby("StockCode")["price"].transform("mean")

Margin assumption

In [12]:
COST_RATIO = 0.60   # assumed unit cost = 60% of the product's median price
panel["unit_cost"] = COST_RATIO * panel.groupby("StockCode")["price"].transform("median")
panel["margin"] = (panel["price"] - panel["unit_cost"]) * panel["units"]

In [13]:
   print(panel[panel["category"] == "Other"].drop_duplicates("StockCode")["desc"].head(30).tolist())

['INFLATABLE POLITICAL GLOBE', 'MINI FUNKY DESIGN TAPES', 'SANDALWOOD FAN', 'SMALL FOLDING SCISSOR(POINTED EDGE)', 'TEATIME PUSH DOWN RUBBER', 'PINK STRAWBERRY HANDBAG', 'RATTLE SNAKE EGGS', 'TRANSPARENT ACRYLIC TAPE DISPENSER', 'ORIGAMI LAVENDER INCENSE/CANDL SET', 'ORIGAMI SANDLEWOOD INCENSE/CAND SET', 'ORIGAMI ROSE INCENSE IN TUBE', 'FAIRY DREAMS INCENSE', 'DRAGONS BLOOD INCENSE', "ASS COL LARGE SAND FROG P'WEIGHT", 'WHITE AND BLUE CERAMIC OIL BURNER', 'BLUE POLKADOT PASSPORT COVER', 'CHERRY BLOSSOM PASSPORT COVER', 'FIRST CLASS PASSPORT COVER', 'QUEEN OF THE SKIES PASSPORT COVER', 'TROPICAL PASSPORT COVER', 'ECONOMY PASSPORT COVER', 'VIPPASSPORT COVER', 'BLUE POLKADOT LUGGAGE TAG', 'CHERRY BLOSSOM LUGGAGE TAG', 'FIRST CLASS LUGGAGE TAG', 'QUEEN OF SKIES LUGGAGE TAG', 'TROPICAL LUGGAGE TAG', 'RED SPOTTY LUGGAGE TAG', 'ECONOMY LUGGAGE TAG', 'STRAWBERRY DREAM CHILDS UMBRELLA']


In [14]:
panel.to_parquet("../data/panel.parquet", index=False)